# PikkuFolding parameter estimates

The notebook considers total witness sizes of $2^{25}$, $2^{27}$, and $2^{29}$ coefficients over $\mathbb Z_q$.  It reports the smallest ring commitment rank whose classical lattice-estimator cost is at least $2^{129}$ and the knowledge error of one fold.  For each ring degree, it uses the fixed-weight sampler targeting the middle, approximately $2^{-100}$, error level.

The theoretical parameters (TP) use the prime $q=2^{64}-59$, coefficient bound $B=2^{10}$, $32$ folding rounds, $a=2$, $e=\varphi/2$, and $\epsilon_{\mathcal C}=0$. The practical parameters (PP) concern only $\varphi=128$ and use $q=2^{50}-2687$, coefficient bound $B=2^3$, $16$ folding rounds, $a=e=2$, and the heuristic $\epsilon_{\mathcal C}=\varphi/(e q^e)$.

In [1]:
import contextlib
import io
import subprocess
import sys
from pathlib import Path

from IPython.display import display
from sage.all import Integer, Mod, RealField, log, oo, sqrt, table

ESTIMATOR_PATH = Path.cwd() / "lattice-estimator"
if not ESTIMATOR_PATH.is_dir():
    raise RuntimeError("Run this notebook from the repository root with ./lattice-estimator checked out.")
sys.path.insert(0, str(ESTIMATOR_PATH))

from estimator import Logging, SIS
import estimator.sis_large_norm as sis_large_norm_module

Logging.set_level(Logging.QUIET)
sis_large_norm_module.profile_precision = 1

ESTIMATOR_COMMIT = subprocess.check_output(
    ["git", "-C", str(ESTIMATOR_PATH), "rev-parse", "HEAD"], text=True
).strip()
print("lattice-estimator commit:", ESTIMATOR_COMMIT)

lattice-estimator commit: 53da5982597709ba0fdf94ea37a84d822310fd84


## Bound and knowledge-error derivation

Each fold has two fresh inputs and three JL layers. The theoretical parameters use $32$ folds, while the practical parameters use $16$. Let $M$ be the number of $\mathbb Z_q$ coordinates, and sample each witness coefficient uniformly from $\{-B,\ldots,B\}$. Then

$$
\mathbb E[x^2]=\frac{B(B+1)}3,\qquad
\mathbb E\|\mathbf v\|_2
\leq\sqrt{\mathbb E\|\mathbf v\|_2^2}
=\sqrt{M\frac{B(B+1)}3}
=: \beta_{\mathsf{in}},\qquad
\beta'_{\mathsf{in}}=\sqrt{2}\,\beta_{\mathsf{in}}
  \left(\frac{u}{\ell}\right)^3.
$$

This replaces the worst-case witness bound $B\sqrt M$. For an honest fold, the inputs are fixed before the centred challenge $c=\sum_i c_iX^i$ is sampled, so the cross terms vanish in expectation:

$$
\mathbb E_c\|\mathbf v_0+c\mathbf v_1\|_2^2
=\beta^2+\mathbb E\|c\|_2^2\gamma^2.
$$

For a fixed-weight challenge of weight $s$, with nonzero coefficients uniform in $\{\pm1,\ldots,\pm B_{\mathcal C}\}$, we use

$$
\tau_\varphi^2
=s\,\mathbb E[c_i^2]
=s\frac{(B_{\mathcal C}+1)(2B_{\mathcal C}+1)}6,
\qquad
H_T
=\sqrt{Tk\tau_\varphi^2\beta_{\mathsf{in}}^2}.
$$

$H_T$ is the honest final-accumulator scale used in the tables. An enforced cap $\widehat H_T$ must include completeness slack. Given this cap, extraction through the previous $T-1$ folds is deterministic:

$$
B_{\mathsf{back}}
=\widehat H_T+(T-1)k\Gamma_\varphi\beta'_{\mathsf{in}}.
$$

For forked challenges $\Delta=c-c'$, we use $\|\Delta\|_{\mathsf{op}}\leq2\Gamma_\varphi$. Only witness sampling and honest folding use averages; extraction and SIS binding use deterministic operator-norm bounds.

The knowledge error for one fold is

$$
\kappa=
\frac{k}{|\mathcal C|}+k\epsilon_{\mathcal C}
+\kappa_{\mathsf{proj}}
+\left(\frac{\log_2N_d}{q}\right)^\mu
+\frac1{q^e}
+\frac{2\sum_{j\in[d]}\nu_j+1}{q^a}.
$$

Here $N_d=256$, $\mu=2$, each JL block fails with probability $2^{-128}$, and $\kappa_{\mathsf{proj}}$ counts all repeated blocks in the two coarse layers.

A uniform scalar uses, on average,

$$
\overline b_q=\frac2q\sum_{x=1}^{(q-1)/2}
  (\lfloor\log_2x\rfloor+1).
$$

We model commitments, batched projections, sumcheck coefficients, and terminal evaluations as uniform. For the shorter projection-trace coordinates, we use

$$
\overline b_{\mathsf{tr}}
=29.26+\log_2(B/2^4)+\tfrac12\log_2(M/2^{25}),
$$

capped at $\overline b_q$. This follows the Gaussian scaling in $B\sqrt M$ and matches the two measured fold sizes within $0.02$ KB. Excluding the local output witness,

$$
\mathsf{bits}_{\mathsf{fold}}
=N_d\overline b_{\mathsf{tr}}
+\mu\varphi\overline b_q
+2aL_{\mathsf{sc}}\overline b_q
+(k+1)\varphi\overline b_q.
$$

The $k$ fresh commitments add $kn\varphi\overline b_q$ bits and are reported separately. The accumulator commitment is already an input and is not sent again.

In [2]:
RF = RealField(200)

RING_DEGREES = (Integer(64), Integer(128), Integer(256))
TOTAL_ZQ_COORDINATE_COUNTS = tuple(Integer(2)**ell for ell in (25, 27, 29))
TARGET_SIS_BITS = RF(129)
MAX_RING_RANK = Integer(128)

K = Integer(2)
FOLDING_ROUNDS = Integer(32)
LAYERS = Integer(3)
JL_HEIGHT = Integer(256)
JL_LAMBDA = Integer(128)
MU = Integer(2)
TRACE_REFERENCE_BITS = RF("29.26")
TRACE_REFERENCE_BOUND = Integer(2)**4
TRACE_REFERENCE_COORDINATES = Integer(2)**25
JL_ALPHA = RF("27.37")
JL_BETA = RF("343.2")
JL_MODULUS_LOSS = Integer(126)
ELL = sqrt(JL_ALPHA)
U = sqrt(JL_BETA)
JL_TIGHTNESS = (U / ELL)**LAYERS
# JL_TIGHTNESS = 1
# T = 64 rows targeting cardinality error 2^-100 in Table 3.
SAMPLER_CARDINALITY_TARGET_BITS = Integer(100)
SAMPLER_TARGET_TRIALS = Integer(64)
SAMPLERS = {
    Integer(64): {
        "s": Integer(25), "B": Integer(2), "Gamma": RF("11.940"),
        "log2_cardinality": RF("102.475"),
    },
    Integer(128): {
        "s": Integer(23), "B": Integer(1), "Gamma": RF("8.357"),
        "log2_cardinality": RF("100.545"),
    },
    Integer(256): {
        "s": Integer(18), "B": Integer(1), "Gamma": RF("8.007"),
        "log2_cardinality": RF("102.578"),
    },
}

def base_sampler_log2_cardinality(phi, s, coefficient_bound):
    return (
        log(RF(binomial(phi, s)), 2)
        + RF(s) * log(RF(2 * coefficient_bound), 2)
    )

required_base_bits = (
    SAMPLER_CARDINALITY_TARGET_BITS
    + log(RF(SAMPLER_TARGET_TRIALS), 2)
)
for phi, sampler in SAMPLERS.items():
    s = sampler["s"]
    coefficient_bound = sampler["B"]
    assert base_sampler_log2_cardinality(phi, s, coefficient_bound) >= required_base_bits
    assert s == 1 or base_sampler_log2_cardinality(phi, s - 1, coefficient_bound) < required_base_bits
    if coefficient_bound > 1:
        assert max(
            base_sampler_log2_cardinality(phi, weight, coefficient_bound - 1)
            for weight in range(1, int(phi) + 1)
        ) < required_base_bits
    assert sampler["log2_cardinality"] >= SAMPLER_CARDINALITY_TARGET_BITS

def challenge_coefficient_energy(phi):
    sampler = SAMPLERS[phi]
    coefficient_bound = sampler["B"]
    mean_nonzero_square = RF(
        (coefficient_bound + 1) * (2 * coefficient_bound + 1)
    ) / 6
    return RF(sampler["s"]) * mean_nonzero_square

def witness_coefficient_energy(input_coefficient_bound):
    bound = RF(input_coefficient_bound)
    return bound * (bound + 1) / 3

def honest_accumulator_bound(
    phi, accumulator_norm, fresh_norm, fresh_count, rounds
):
    return sqrt(
        RF(accumulator_norm)**2
        + rounds * fresh_count * challenge_coefficient_energy(phi)
        * RF(fresh_norm)**2
    )

def backward_extraction_bound(
    phi, output_bound, fresh_norm, fresh_count, rounds
):
    return (
        RF(output_bound)
        + rounds * fresh_count * SAMPLERS[phi]["Gamma"] * fresh_norm
    )

def extracted_sis_bounds(
    phi, total_zq_coordinates, input_coeff_infinity_bound, folding_rounds
):
    assert total_zq_coordinates % phi == 0
    m = total_zq_coordinates // phi
    gamma = SAMPLERS[phi]["Gamma"]
    tau = sqrt(challenge_coefficient_energy(phi))

    beta_in = sqrt(
        RF(phi * m) * witness_coefficient_energy(input_coeff_infinity_bound)
    )
    beta_projection_before_fine = sqrt(RF(K)) * beta_in * U**(LAYERS - 1)
    beta_in_extracted = sqrt(RF(K)) * beta_in * JL_TIGHTNESS
    beta_out_last_round = honest_accumulator_bound(
        phi, RF(0), beta_in, K, folding_rounds
    )

    extraction_before_first_round = backward_extraction_bound(
        phi, beta_out_last_round, beta_in_extracted, K,
        folding_rounds - 1,
    )
    extracted_accumulator = backward_extraction_bound(
        phi, extraction_before_first_round, beta_in_extracted, K, 1
    )

    beta_binding = (K + 1) * extraction_before_first_round * (2 * gamma)**K
    rho_binding = (2 * gamma)**K
    beta_sis_accumulator = 2 * beta_binding * rho_binding
    beta_sis_fresh = 8 * gamma * extraction_before_first_round

    return {
        "m": m,
        "gamma": gamma,
        "tau": tau,
        "beta_in": beta_in,
        "beta_projection_before_fine": beta_projection_before_fine,
        "beta_in_extracted": beta_in_extracted,
        "beta_out_last_round": beta_out_last_round,
        "extracted_accumulator": extracted_accumulator,
        "beta_sis_fresh": beta_sis_fresh,
        "beta_sis_accumulator": beta_sis_accumulator,
        "beta_sis": max(beta_sis_fresh, beta_sis_accumulator),
    }

def balanced_projection_parameters(phi, total_zq_coordinates):
    m = total_zq_coordinates // phi
    nu_0 = Integer(log(K * m, 2))
    nu_fine = Integer(log(JL_HEIGHT, 2))
    coarse_layers = LAYERS - 1
    quotient, remainder = (nu_0 - nu_fine).quo_rem(coarse_layers)
    shrinkages = [quotient + (1 if j < remainder else 0) for j in range(coarse_layers)]

    nus = [nu_0]
    for shrinkage in shrinkages:
        nus.append(nus[-1] - shrinkage)
    assert nus[-1] == nu_fine

    repetitions = [
        Integer(2)**(nus[j + 1] - nu_fine) for j in range(coarse_layers)
    ] + [Integer(1)]
    etas = [phi * repetitions[j] for j in range(coarse_layers)] + [Integer(1)]
    return nus, repetitions, etas

def average_uniform_scalar_bits(q):
    half = (q - 1) // 2
    top_bits = half.nbits()
    total = sum(Integer(bits) * Integer(2)**(bits - 1) for bits in range(1, top_bits))
    total += top_bits * (half - Integer(2)**(top_bits - 1) + 1)
    return RF(2 * total) / q

def proof_size_kb(
    phi, total_zq_coordinates, q, a, input_coeff_infinity_bound, ring_rank
):
    uniform_bits = average_uniform_scalar_bits(q)
    trace_bits = (
        TRACE_REFERENCE_BITS
        + log(RF(input_coeff_infinity_bound) / TRACE_REFERENCE_BOUND, 2)
        + RF(1) / 2 * log(RF(total_zq_coordinates) / TRACE_REFERENCE_COORDINATES, 2)
    )
    trace_bits = min(trace_bits, uniform_bits)
    nus, _, _ = balanced_projection_parameters(phi, total_zq_coordinates)
    sumcheck_rounds = sum(nus)

    projection_bits = JL_HEIGHT * trace_bits + MU * phi * uniform_bits
    round_polynomial_bits = 2 * sumcheck_rounds * a * uniform_bits
    terminal_bits = (K + 1) * phi * uniform_bits
    fold_bits = projection_bits + round_polynomial_bits + terminal_bits
    commitment_bits = K * ring_rank * phi * uniform_bits
    bits_per_kb = RF(8 * 1024)
    return commitment_bits / bits_per_kb, fold_bits / bits_per_kb

def knowledge_error(phi, total_zq_coordinates, q, a, e, epsilon_challenge):
    nus, _, etas = balanced_projection_parameters(phi, total_zq_coordinates)
    return (
        K / RF(2)**SAMPLERS[phi]["log2_cardinality"]
        + K * RF(epsilon_challenge)
        + sum(etas) * RF(2)**(-JL_LAMBDA)
        + (RF(log(JL_HEIGHT, 2)) / RF(q))**MU
        + RF(q)**(-e)
        + RF(2 * sum(nus) + 1) / RF(q)**a
    )

def sis_security_bits(phi, q, beta_sis, ring_rank):
    params = SIS.Parameters(
        n=phi * ring_rank,
        q=q,
        length_bound=beta_sis,
        norm=2,
    )
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            costs = SIS.estimate(params, quiet=True)
    except Exception:
        return -oo

    attack_bits = [log(cost["rop"], 2) for cost in costs.values() if "rop" in cost]
    return min(attack_bits) if attack_bits else -oo

def smallest_secure_rank(phi, q, beta_sis):
    if beta_sis >= q:
        return None, None
    for ring_rank in range(1, MAX_RING_RANK + 1):
        security_bits = sis_security_bits(phi, q, beta_sis, ring_rank)
        if security_bits >= TARGET_SIS_BITS:
            return Integer(ring_rank), security_bits
    return None, None

def estimate_scenario(
    moduli, input_coeff_infinity_bound, ring_degrees, folding_rounds,
    a, e_by_degree,
    epsilon_function,
):
    rows = []

    for total_zq_coordinates in TOTAL_ZQ_COORDINATE_COUNTS:
        for phi in ring_degrees:
            q = moduli[phi] if isinstance(moduli, dict) else moduli
            e = e_by_degree[phi]
            assert q.is_prime()
            assert e % a == 0
            assert Mod(q, 2 * phi).multiplicative_order() == e

            bounds = extracted_sis_bounds(
                phi, total_zq_coordinates, input_coeff_infinity_bound,
                folding_rounds,
            )
            beta_sis = bounds["beta_sis"]
            assert bounds["beta_projection_before_fine"] < RF(q) / JL_MODULUS_LOSS
            rank, _ = smallest_secure_rank(phi, q, beta_sis)

            if rank is None:
                rank_entry = "--"
                commitment_kb_entry = "--"
                fold_kb_entry = "--"
                status = "insecure"
            else:
                rank_entry = rank
                commitment_kb, fold_kb = proof_size_kb(
                    phi, total_zq_coordinates, q, a,
                    input_coeff_infinity_bound, rank,
                )
                commitment_kb_entry = f"{commitment_kb:.2f}"
                fold_kb_entry = f"{fold_kb:.2f}"
                status = "secure"

            epsilon_challenge = epsilon_function(phi, q, e)
            kappa = knowledge_error(
                phi, total_zq_coordinates, q, a, e, epsilon_challenge
            )
            rows.append([
                f"2^{Integer(log(total_zq_coordinates, 2))}",
                phi,
                f"2^{Integer(log(bounds['m'], 2))}",
                a, e,
                f"{float(bounds['gamma']):.3f}",
                f"{float(bounds['tau']):.3f}",
                f"{float(log(beta_sis, 2)):.3f}",
                rank_entry,
                commitment_kb_entry,
                fold_kb_entry,
                f"{float(log(kappa, 2)):.3f}",
                status,
            ])

    return table(
        [[r"$M$ over $\mathbb Z_q$", r"$\varphi$", r"$m$ in $R_q$",
          r"$a$", r"$e$", r"$\Gamma$", r"$\tau$",
          r"$\log_2\beta_{\mathsf{SIS}}$", "smallest rank",
          "fresh commitments (KB)", "in-protocol (KB)",
          r"$\log_2\kappa$", "status"]] + rows,
        header_row=True,
    )

print(f"log2 JL extraction loss = {float(log(JL_TIGHTNESS, 2)):.6f}")

log2 JL extraction loss = 5.472573


## Theoretical parameters (TP)

In [3]:
q = Integer(2)**64 - 59
B = Integer(2)**10
A = Integer(2)
E_BY_DEGREE = {phi: phi // 2 for phi in RING_DEGREES}

assert q.is_prime() and q % 8 == 5
for phi in RING_DEGREES:
    assert 2 * SAMPLERS[phi]["B"] < sqrt(RF(q) / 2)

def exact_epsilon(phi, q, e):
    return RF(0)

print(f"q = 2^64 - 59 = {q}, input coefficient bound B = 2^10")
tp_table = estimate_scenario(
    q, B, RING_DEGREES, FOLDING_ROUNDS, A, E_BY_DEGREE, exact_epsilon
)
display(tp_table)

q = 2^64 - 59 = 18446744073709551557, input coefficient bound B = 2^10


\(M\) over \(\mathbb Z_q\),\(\varphi\),\(m\) in \(R_q\),\(a\),\(e\),\(\Gamma\),\(\tau\),\(\log_2\beta_{\mathsf{SIS}}\),smallest rank,fresh commitments (KB),in-protocol (KB),\(\log_2\kappa\),status
2^25,\(64\),2^19,\(2\),\(32\),11.940,7.906,58.111,\(33\),31.97,4.80,-101.475,secure
2^25,\(128\),2^18,\(2\),\(64\),8.357,4.796,55.537,\(15\),29.06,7.16,-99.545,secure
2^25,\(256\),2^17,\(2\),\(128\),8.007,4.243,55.228,\(8\),31.00,11.97,-101.578,secure
2^27,\(64\),2^21,\(2\),\(32\),11.940,7.906,59.111,\(34\),32.94,4.92,-101.475,secure
2^27,\(128\),2^20,\(2\),\(64\),8.357,4.796,56.537,\(16\),31.00,7.28,-99.545,secure
2^27,\(256\),2^19,\(2\),\(128\),8.007,4.243,56.228,\(8\),31.00,12.09,-101.578,secure
2^29,\(64\),2^23,\(2\),\(32\),11.940,7.906,60.111,\(35\),33.91,5.04,-101.475,secure
2^29,\(128\),2^22,\(2\),\(64\),8.357,4.796,57.537,\(16\),31.00,7.40,-99.545,secure
2^29,\(256\),2^21,\(2\),\(128\),8.007,4.243,57.228,\(8\),31.00,12.21,-101.577,secure


## Practical parameters (PP)

In [4]:
q = Integer(1125899906839937)  # 2^50 - 2687
B = Integer(2)**3
A = Integer(2)
PP_FOLDING_ROUNDS = Integer(16)
PP_DEGREES = (Integer(128),)
E_BY_DEGREE = {Integer(128): Integer(2)}

assert q.is_prime()

def heuristic_epsilon(phi, q, e):
    return RF(phi) / (e * RF(q)**e)

print(f"q = 2^50 - 2687 = {q}, input coefficient bound B = 2^3")


pp_table = estimate_scenario(
    q, B, PP_DEGREES, PP_FOLDING_ROUNDS, A, E_BY_DEGREE, heuristic_epsilon
)
display(pp_table)

q = 2^50 - 2687 = 1125899906839937, input coefficient bound B = 2^3


\(M\) over \(\mathbb Z_q\),\(\varphi\),\(m\) in \(R_q\),\(a\),\(e\),\(\Gamma\),\(\tau\),\(\log_2\beta_{\mathsf{SIS}}\),smallest rank,fresh commitments (KB),in-protocol (KB),\(\log_2\kappa\),status
2^25,\(128\),2^18,\(2\),\(2\),8.357,4.796,47.574,\(14\),21.00,5.57,-91.895,secure
2^27,\(128\),2^20,\(2\),\(2\),8.357,4.796,48.574,\(15\),22.50,5.67,-91.864,secure
2^29,\(128\),2^22,\(2\),\(2\),8.357,4.796,49.574,\(16\),24.00,5.77,-91.833,secure
